In [2]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
df= pd.read_csv("data/processed_rat_data.csv")

3. TDS vs Avg Feed Intake Split by Gender


What we're plotting:

- X-axis: TDS level (continuous or categorical)
- Y-axis: Average weekly feed intake across all 15 weeks
- Two lines/colors: Male vs Female
Include error bars (standard deviation or confidence intervals)

Key questions:

Is there a "sweet spot" TDS level where feed intake is optimal?
Do males and females have different optimal TDS levels?
Is the relationship linear, U-shaped, or inverted-U?
Do males consume more feed overall (we'd expect yes), but does the pattern differ?

In [3]:

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directory
output_dir = Path('outputs')
# output_dir.mkdir(exist_ok=True)
print(f"Data loaded: {len(df)} rows")
print(f"Unique rats: {df['unique_rat_id'].nunique()}")

# ============================================================================
# STEP 1: Prepare Data
# ============================================================================
print("\n" + "="*80)
print("STEP 1: Data Preparation")
print("="*80)

# Create group_short if not exists
def shorten_group_name(name):
    if 'Group 1' in name and 'RO water with < 20' in name:
        return 'G01: RO <20 TDS'
    elif 'Group 2' in name:
        return 'G02: RO 50-75 TDS'
    elif 'Group 3' in name and 'alternate' not in name.lower():
        return 'G03: RO 125-150 TDS'
    elif 'Group 4' in name:
        return 'G04: Telugu Ganga'
    elif 'Group 5' in name and 'alternate' not in name.lower():
        return 'G05: Kalyani Dam'
    elif 'Group 6' in name and 'alternate' not in name.lower():
        return 'G06: Ground Water'
    elif 'Group 7' in name:
        return 'G07: RO <20 (Fasting)'
    elif 'Group 8' in name:
        return 'G08: Kalyani (Fasting)'
    elif 'Group 9' in name:
        return 'G09: BIS Standard'
    elif 'Group 10' in name:
        return 'G10: Ground (Fasting)'
    elif 'Group 11' in name:
        return 'G11: RO 125-150 (Fasting)'
    return name

if 'group_short' not in df.columns:
    df['group_short'] = df['water_group'].apply(shorten_group_name)

# Calculate average weekly feed intake per rat across all 15 weeks
print("\nCalculating average weekly feed intake per rat...")
avg_feed_by_rat = df.groupby(['unique_rat_id', 'gender', 'group_short'])['weekly_feed_intake'].mean().reset_index()
avg_feed_by_rat.rename(columns={'weekly_feed_intake': 'avg_weekly_feed'}, inplace=True)

print(f"Rats analyzed: {len(avg_feed_by_rat)}")
print(f"Groups: {sorted(avg_feed_by_rat['group_short'].unique())}")

# Calculate summary statistics by group and gender
print("\nCalculating summary statistics...")
summary = avg_feed_by_rat.groupby(['group_short', 'gender'])['avg_weekly_feed'].agg([
    'mean', 'std', 'sem', 'count', 'min', 'max'
]).reset_index()

# Calculate confidence intervals (95%)
summary['ci_lower'] = summary['mean'] - 1.96 * summary['sem']
summary['ci_upper'] = summary['mean'] + 1.96 * summary['sem']

print("\n--- Summary Statistics ---")
print(summary.to_string())

# Save summary to CSV
summary.to_csv(output_dir / 'feed_intake_by_gender_summary.csv', index=False)
print(f"\n✓ Saved: feed_intake_by_gender_summary.csv")

# ============================================================================
# STEP 2: Overall Gender Comparison (Question 4a)
# ============================================================================
print("\n" + "="*80)
print("STEP 2: Overall Gender Comparison")
print("="*80)

male_avg = avg_feed_by_rat[avg_feed_by_rat['gender'] == 'male']['avg_weekly_feed']
female_avg = avg_feed_by_rat[avg_feed_by_rat['gender'] == 'female']['avg_weekly_feed']

print(f"\nMale average feed intake: {male_avg.mean():.2f} ± {male_avg.std():.2f} g/week")
print(f"Female average feed intake: {female_avg.mean():.2f} ± {female_avg.std():.2f} g/week")
print(f"Difference: {male_avg.mean() - female_avg.mean():.2f} g/week")

# Statistical test
t_stat, p_value = stats.ttest_ind(male_avg, female_avg)
print(f"\nT-test: t={t_stat:.3f}, p={p_value:.4f}")
if p_value < 0.05:
    print("✓ Statistically significant difference (p < 0.05)")
else:
    print("✗ Not statistically significant (p >= 0.05)")

# ============================================================================
# STEP 3: Identify Optimal Groups (Questions 1 & 2)
# ============================================================================
print("\n" + "="*80)
print("STEP 3: Optimal Water Groups")
print("="*80)

# Overall optimal (lowest feed intake = most efficient if weight gain maintained)
optimal_overall = summary.loc[summary['mean'].idxmin()]
print(f"\nLowest overall feed intake:")
print(f"  Group: {optimal_overall['group_short']}")
print(f"  Gender: {optimal_overall['gender']}")
print(f"  Mean: {optimal_overall['mean']:.2f} g/week")

# Highest feed intake
highest_overall = summary.loc[summary['mean'].idxmax()]
print(f"\nHighest overall feed intake:")
print(f"  Group: {highest_overall['group_short']}")
print(f"  Gender: {highest_overall['gender']}")
print(f"  Mean: {highest_overall['mean']:.2f} g/week")

# Optimal by gender
print("\n--- Optimal Groups by Gender ---")
male_summary = summary[summary['gender'] == 'male']
female_summary = summary[summary['gender'] == 'female']

male_optimal = male_summary.loc[male_summary['mean'].idxmin()]
female_optimal = female_summary.loc[female_summary['mean'].idxmin()]

print(f"\nMale optimal: {male_optimal['group_short']} ({male_optimal['mean']:.2f} g/week)")
print(f"Female optimal: {female_optimal['group_short']} ({female_optimal['mean']:.2f} g/week)")

if male_optimal['group_short'] == female_optimal['group_short']:
    print("✓ Same optimal group for both genders")
else:
    print("✗ Different optimal groups for males and females")

# ============================================================================
# STEP 4: Pattern Analysis (Question 3)
# ============================================================================
print("\n" + "="*80)
print("STEP 4: Pattern Analysis")
print("="*80)

# Get group order (G01 -> G11)
groups_ordered = sorted(summary['group_short'].unique())

print("\n--- Feed Intake by Group (Male) ---")
male_by_group = male_summary.sort_values('group_short')
for _, row in male_by_group.iterrows():
    print(f"{row['group_short']}: {row['mean']:.2f} ± {row['std']:.2f} g/week")

print("\n--- Feed Intake by Group (Female) ---")
female_by_group = female_summary.sort_values('group_short')
for _, row in female_by_group.iterrows():
    print(f"{row['group_short']}: {row['mean']:.2f} ± {row['std']:.2f} g/week")

# Pattern classification
def classify_pattern(values):
    """Classify if pattern is linear, U-shaped, inverted-U, or irregular"""
    if len(values) < 3:
        return "insufficient data"
    
    # Check for monotonic increase/decrease
    increasing = all(values[i] <= values[i+1] for i in range(len(values)-1))
    decreasing = all(values[i] >= values[i+1] for i in range(len(values)-1))
    
    if increasing:
        return "linear increasing"
    elif decreasing:
        return "linear decreasing"
    
    # Find min and max positions
    min_idx = values.index(min(values))
    max_idx = values.index(max(values))
    
    # U-shaped: min in middle, max at ends
    if 2 <= min_idx <= len(values)-3:
        return "U-shaped (min in middle)"
    
    # Inverted-U: max in middle, min at ends
    if 2 <= max_idx <= len(values)-3:
        return "inverted-U (max in middle)"
    
    return "irregular/no clear pattern"

male_values = male_by_group['mean'].tolist()
female_values = female_by_group['mean'].tolist()

print(f"\n--- Pattern Classification ---")
print(f"Male pattern: {classify_pattern(male_values)}")
print(f"Female pattern: {classify_pattern(female_values)}")

# ============================================================================
# STEP 5: Create Visualizations
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Creating Visualizations")
print("="*80)

# Create figure with multiple panels
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# ============================================================================
# PANEL 1: Line Plot with Error Bars (Main Plot)
# ============================================================================
print("\nCreating Panel 1: Line plot with error bars...")
ax1 = fig.add_subplot(gs[0, :])

# Prepare data for plotting
male_summary_sorted = male_summary.sort_values('group_short')
female_summary_sorted = female_summary.sort_values('group_short')

x_positions = range(len(groups_ordered))

# Plot male line
ax1.errorbar(x_positions, 
             male_summary_sorted['mean'].values,
             yerr=male_summary_sorted['sem'].values,
             marker='o', 
             markersize=10,
             linewidth=2.5,
             capsize=5,
             capthick=2,
             label='Male',
             color='#2E86AB',
             alpha=0.8)

# Plot female line
ax1.errorbar(x_positions, 
             female_summary_sorted['mean'].values,
             yerr=female_summary_sorted['sem'].values,
             marker='^', 
             markersize=10,
             linewidth=2.5,
             capsize=5,
             capthick=2,
             label='Female',
             color='#E63946',
             alpha=0.8)

ax1.set_xlabel('Water Group', fontsize=14, fontweight='bold')
ax1.set_ylabel('Average Weekly Feed Intake (g)', fontsize=14, fontweight='bold')
ax1.set_title('Average Feed Intake by Water Group and Gender\n(Error bars: ±SEM)', 
              fontsize=16, fontweight='bold', pad=20)
ax1.set_xticks(x_positions)
ax1.set_xticklabels(groups_ordered, rotation=45, ha='right', fontsize=10)
ax1.legend(fontsize=12, loc='best', framealpha=0.9)
ax1.grid(True, alpha=0.3)

# Add horizontal line at overall mean
overall_mean = avg_feed_by_rat['avg_weekly_feed'].mean()
ax1.axhline(y=overall_mean, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax1.text(len(groups_ordered)-1, overall_mean, f'  Overall Mean: {overall_mean:.1f}g', 
         va='center', ha='right', fontsize=9, color='gray')

# ============================================================================
# PANEL 2: Box Plot by Group
# ============================================================================
print("Creating Panel 2: Box plots...")
ax2 = fig.add_subplot(gs[1, 0])

# Prepare data for box plot
plot_data = []
plot_labels = []
plot_colors = []

for group in groups_ordered:
    male_data = avg_feed_by_rat[(avg_feed_by_rat['group_short'] == group) & 
                                 (avg_feed_by_rat['gender'] == 'male')]['avg_weekly_feed']
    female_data = avg_feed_by_rat[(avg_feed_by_rat['group_short'] == group) & 
                                   (avg_feed_by_rat['gender'] == 'female')]['avg_weekly_feed']
    
    if len(male_data) > 0:
        plot_data.append(male_data)
        plot_labels.append(f"{group}\n(M)")
        plot_colors.append('#2E86AB')
    
    if len(female_data) > 0:
        plot_data.append(female_data)
        plot_labels.append(f"{group}\n(F)")
        plot_colors.append('#E63946')

bp = ax2.boxplot(plot_data, 
                  labels=plot_labels,
                  patch_artist=True,
                  showfliers=True,
                  flierprops=dict(marker='o', markersize=4, alpha=0.5))

# Color the boxes
for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax2.set_ylabel('Average Weekly Feed Intake (g)', fontsize=12, fontweight='bold')
ax2.set_title('Distribution by Group and Gender', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', labelsize=7, rotation=90)
ax2.grid(True, alpha=0.3, axis='y')

# ============================================================================
# PANEL 3: Bar Chart - Mean Comparison
# ============================================================================
print("Creating Panel 3: Bar chart comparison...")
ax3 = fig.add_subplot(gs[1, 1])

x = np.arange(len(groups_ordered))
width = 0.35

male_means = male_summary_sorted['mean'].values
female_means = female_summary_sorted['mean'].values
male_sems = male_summary_sorted['sem'].values
female_sems = female_summary_sorted['sem'].values

bars1 = ax3.bar(x - width/2, male_means, width, 
                yerr=male_sems,
                label='Male', 
                color='#2E86AB', 
                alpha=0.7,
                capsize=3)

bars2 = ax3.bar(x + width/2, female_means, width,
                yerr=female_sems,
                label='Female',
                color='#E63946',
                alpha=0.7,
                capsize=3)

ax3.set_ylabel('Average Weekly Feed Intake (g)', fontsize=12, fontweight='bold')
ax3.set_title('Mean Feed Intake Comparison', fontsize=13, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(groups_ordered, rotation=45, ha='right', fontsize=9)
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3, axis='y')

# ============================================================================
# Save Figure
# ============================================================================
plt.suptitle('Feed Intake Analysis by Water Group and Gender', 
             fontsize=18, fontweight='bold', y=0.995)

output_path = output_dir / 'feed_intake_by_gender_analysis.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Saved: {output_path}")
plt.close()

# ============================================================================
# STEP 6: Additional Analysis - Gender Interaction
# ============================================================================
print("\n" + "="*80)
print("STEP 6: Gender × Group Interaction Analysis")
print("="*80)

# Calculate difference between male and female for each group
interaction_data = []
for group in groups_ordered:
    male_mean = male_summary[male_summary['group_short'] == group]['mean'].values[0]
    female_mean = female_summary[female_summary['group_short'] == group]['mean'].values[0]
    difference = male_mean - female_mean
    interaction_data.append({
        'group': group,
        'male_mean': male_mean,
        'female_mean': female_mean,
        'difference': difference,
        'percent_diff': (difference / female_mean) * 100
    })

interaction_df = pd.DataFrame(interaction_data)
interaction_df.to_csv(output_dir / 'gender_interaction_analysis.csv', index=False)

print("\n--- Male - Female Difference by Group ---")
print(interaction_df.to_string(index=False))

# Check if interaction is consistent
diff_std = interaction_df['difference'].std()
diff_mean = interaction_df['difference'].mean()
cv = (diff_std / diff_mean) * 100  # Coefficient of variation

print(f"\nDifference statistics:")
print(f"  Mean difference: {diff_mean:.2f} g/week")
print(f"  SD of differences: {diff_std:.2f}")
print(f"  Coefficient of variation: {cv:.1f}%")

if cv < 20:
    print("  → Consistent pattern (parallel lines) - no strong interaction")
else:
    print("  → Variable pattern (non-parallel lines) - interaction present")

# ============================================================================
# STEP 7: Correlation with FCR
# ============================================================================
print("\n" + "="*80)
print("STEP 7: Relationship with FCR Performance")
print("="*80)

# Rank groups by FCR (from FCR analysis) and feed intake
fcr_ranking = {
    'G03: RO 125-150 TDS': 1,
    'G09: BIS Standard': 2,
    'G08: Kalyani (Fasting)': 3,
    'G06: Ground Water': 4,
    'G10: Ground (Fasting)': 5,
    'G11: RO 125-150 (Fasting)': 6,
    'G07: RO <20 (Fasting)': 7,
    'G04: Telugu Ganga': 8,
    'G05: Kalyani Dam': 9,
    'G01: RO <20 TDS': 10,
    'G02: RO 50-75 TDS': 11
}

# Add FCR rank to summary
summary['fcr_rank'] = summary['group_short'].map(fcr_ranking)

# Calculate correlation
correlation_data = summary.groupby('group_short').agg({
    'mean': 'mean',  # Average of male and female
    'fcr_rank': 'first'
}).reset_index()

corr_coef = correlation_data['mean'].corr(correlation_data['fcr_rank'])
print(f"\nCorrelation between feed intake and FCR rank: {corr_coef:.3f}")

if abs(corr_coef) > 0.7:
    direction = "positive" if corr_coef > 0 else "negative"
    print(f"  → Strong {direction} correlation")
elif abs(corr_coef) > 0.3:
    direction = "positive" if corr_coef > 0 else "negative"
    print(f"  → Moderate {direction} correlation")
else:
    print(f"  → Weak correlation")

print(f"\nInterpretation:")
if corr_coef > 0.3:
    print("  Groups with higher feed intake tend to have worse FCR (less efficient)")
elif corr_coef < -0.3:
    print("  Groups with higher feed intake tend to have better FCR (more efficient)")
else:
    print("  Feed intake and FCR are largely independent")

# ============================================================================
# STEP 8: Generate Final Summary Report
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n" + "="*50)
print("QUESTION 1: Sweet Spot Water Group")
print("="*50)
print(f"Lowest feed intake: {optimal_overall['group_short']} ({optimal_overall['gender']})")
print(f"  Mean: {optimal_overall['mean']:.2f} g/week")
print(f"\nHighest feed intake: {highest_overall['group_short']} ({highest_overall['gender']})")
print(f"  Mean: {highest_overall['mean']:.2f} g/week")

print("\n" + "="*50)
print("QUESTION 2: Gender-Specific Optima")
print("="*50)
print(f"Male optimal: {male_optimal['group_short']} ({male_optimal['mean']:.2f} g/week)")
print(f"Female optimal: {female_optimal['group_short']} ({female_optimal['mean']:.2f} g/week)")
if male_optimal['group_short'] == female_optimal['group_short']:
    print("✓ Same optimal group for both genders")
else:
    print("✗ Different optimal groups")

print("\n" + "="*50)
print("QUESTION 3: Pattern Shape")
print("="*50)
print(f"Male pattern: {classify_pattern(male_values)}")
print(f"Female pattern: {classify_pattern(female_values)}")

print("\n" + "="*50)
print("QUESTION 4: Gender Differences")
print("="*50)
print(f"Male average: {male_avg.mean():.2f} g/week")
print(f"Female average: {female_avg.mean():.2f} g/week")
print(f"Difference: {male_avg.mean() - female_avg.mean():.2f} g/week ({((male_avg.mean() - female_avg.mean())/female_avg.mean()*100):.1f}%)")
print(f"Statistical significance: p = {p_value:.4f}")
print(f"\nPattern consistency (CV): {cv:.1f}%")
if cv < 20:
    print("  → Patterns are similar (parallel)")
else:
    print("  → Patterns differ (interaction)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  1. feed_intake_by_gender_analysis.png - Main visualization")
print("  2. feed_intake_by_gender_summary.csv - Summary statistics")
print("  3. gender_interaction_analysis.csv - Interaction analysis")
print("\n" + "="*80)

Data loaded: 1650 rows
Unique rats: 110

STEP 1: Data Preparation

Calculating average weekly feed intake per rat...
Rats analyzed: 110
Groups: ['G01: RO <20 TDS', 'G02: RO 50-75 TDS', 'G03: RO 125-150 TDS', 'G04: Telugu Ganga', 'G05: Kalyani Dam', 'G06: Ground Water', 'G07: RO <20 (Fasting)', 'G08: Kalyani (Fasting)', 'G09: BIS Standard', 'G10: Ground (Fasting)', 'G11: RO 125-150 (Fasting)']

Calculating summary statistics...

--- Summary Statistics ---
                  group_short  gender       mean       std       sem  count        min        max   ci_lower   ci_upper
0             G01: RO <20 TDS  female  18.312613  0.609693  0.272663      5  17.373800  18.819667  17.778194  18.847033
1             G01: RO <20 TDS    male  19.044773  0.851618  0.380855      5  17.926067  20.222467  18.298297  19.791249
2           G02: RO 50-75 TDS  female  16.456467  0.721181  0.322522      5  15.608133  17.420133  15.824324  17.088610
3           G02: RO 50-75 TDS    male  17.830493  1.156937  0

/var/folders/h9/24vbd1m931s_2j_jt_vy8cyr0000gn/T/ipykernel_21793/629926728.py:278: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot(plot_data,



✓ Saved: outputs/feed_intake_by_gender_analysis.png

STEP 6: Gender × Group Interaction Analysis

--- Male - Female Difference by Group ---
                    group  male_mean  female_mean  difference  percent_diff
          G01: RO <20 TDS  19.044773    18.312613    0.732160      3.998119
        G02: RO 50-75 TDS  17.830493    16.456467    1.374027      8.349463
      G03: RO 125-150 TDS  19.525893    15.824440    3.701453     23.390738
        G04: Telugu Ganga  19.168640    17.088827    2.079813     12.170603
         G05: Kalyani Dam  19.321213    17.922040    1.399173      7.806998
        G06: Ground Water  16.519067    14.715440    1.803627     12.256695
    G07: RO <20 (Fasting)   9.176627     9.228627   -0.052000     -0.563464
   G08: Kalyani (Fasting)   9.878547    10.051853   -0.173307     -1.724126
        G09: BIS Standard  18.994080    16.373947    2.620133     16.001844
    G10: Ground (Fasting)  10.041480     9.960560    0.080920      0.812404
G11: RO 125-150 (Fastin